# Simple RAG with LangChain (Python 3.11.9)

This notebook loads five realistic, fictional training PDFs from the `sample_pdfs` folder and uses them as a RAG knowledge base. Each PDF has three pages of document text.

**Flow:** load PDFs → split text → index chunks → retrieve relevant text → answer questions.

Keep this notebook beside the `sample_pdfs` folder and run cells from top to bottom. You need an OpenAI API key.

## 1. Install packages and set your API key

Run the install cell once. Restart the kernel if prompted. Add your key to a `.env` file beside this notebook:

```text
OPENAI_API_KEY=your-key-here
```

Keep your real key private.

In [1]:
#%pip install -q langchain langchain-classic langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf python-dotenv pandas

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file or your environment, then rerun this cell.")

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 2. Put the PDF files beside the notebook

Place the provided `sample_pdfs` folder in the same directory as this notebook. It should contain `company_overview.pdf`, `leave_policy.pdf`, `expense_policy.pdf`, `security_policy.pdf`, and `remote_work_policy.pdf`. This notebook reads those files directly; it does not generate them.

## 3. Load all five PDF files

Each page becomes a document, with the PDF filename and page number kept as metadata.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_dir = Path("sample_pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))
if len(pdf_files) != 5:
    raise FileNotFoundError("Expected five PDF files in the sample_pdfs folder. Put that folder beside the notebook.")

pages = []
for pdf_file in pdf_files:
    pages.extend(PyPDFLoader(str(pdf_file)).load())

print("PDF files:", len(pdf_files))
print("Pages loaded:", len(pages))
for pdf_file in pdf_files:
    page_count = sum(1 for page in pages if Path(page.metadata["source"]).name == pdf_file.name)
    print(f"{pdf_file.name}: {page_count} pages")
    assert page_count >= 2, f"{pdf_file.name} should have at least two pages"


C:\Users\Admin\AppData\Local\Temp\ipykernel_8684\2218931891.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


PDF files: 5
Pages loaded: 15
company_overview.pdf: 3 pages
expense_policy.pdf: 3 pages
leave_policy.pdf: 3 pages
remote_work_policy.pdf: 3 pages
security_policy.pdf: 3 pages


## 4. Split documents into chunks

Chunking breaks each PDF into smaller pieces that can be searched.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print("Chunks created:", len(chunks))
print("Example chunk:", chunks[0].page_content[:300])

Chunks created: 45
Example chunk: Company Overview
1. Organization and Services
Northstar Services is a fictional professional services company used for training and demonstrations. It
provides technology consulting, application support, and process improvement services to medium and
large organizations. Its offices are located in B


## 5. Create embeddings and store chunks in Chroma

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_documents",
    persist_directory="chroma_db"
)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector database is ready.")

Vector database is ready.


## 6. Build the RAG question-answer chain

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If the answer is missing, say you could not find it.\n\nContext:\n{context}"),
    ("human", "{input}")
])
answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, answer_chain)

## 7. Ask questions about the PDFs

In [7]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source files:", sorted({Path(doc.metadata["source"]).name for doc in result["context"]}))
    print()
    return result

ask("How many paid annual leave days do full-time employees get?")

Question: How many paid annual leave days do full-time employees get?
Answer: Full-time employees receive 18 days of paid annual leave for each calendar year.
Source files: ['leave_policy.pdf']



{'input': 'How many paid annual leave days do full-time employees get?',
 'context': [Document(id='2fa8e3a3-c0c3-45dc-a31e-83ff1bf1d756', metadata={'keywords': '', 'total_pages': 3, 'creator': '(unspecified)', 'creationdate': '2026-10-28T09:15:21+02:00', 'producer': 'ReportLab PDF Library - (opensource)', 'page': 0, 'page_label': '1', 'trapped': '/False', 'subject': '(unspecified)', 'source': 'sample_pdfs\\leave_policy.pdf', 'title': 'Leave Policy', 'author': 'Training example', 'moddate': '2026-10-28T09:15:21+02:00'}, page_content='Leave Policy\n1. Annual and Personal Leave\nIn this fictional training policy, full-time employees receive 18 days of paid annual leave for each\ncalendar year. Annual leave is intended for rest, personal commitments, and planned time away from\nwork. Part-time employees and employees who join or leave during the year should check their\nindividual entitlement with People Operations.\nEmployees should submit planned leave in the leave system at least five w

In [8]:
ask("When should employees submit expense claims?")
ask("What should I do if I receive a phishing message?")
ask("How many days per week can an employee work remotely?")

Question: When should employees submit expense claims?
Answer: Employees should submit expense claims in the expense system within 30 days of the purchase.
Source files: ['expense_policy.pdf']

Question: What should I do if I receive a phishing message?
Answer: If you receive a phishing message, do not click suspicious links, open unexpected attachments, or reply with sensitive information. Use the email client’s report function or forward the message to the IT help desk according to local instructions. Report it promptly so IT can protect data and arrange next steps.
Source files: ['remote_work_policy.pdf', 'security_policy.pdf']

Question: How many days per week can an employee work remotely?
Answer: An employee can work remotely up to two days per week with manager approval.
Source files: ['remote_work_policy.pdf']



{'input': 'How many days per week can an employee work remotely?',
 'context': [Document(id='a8df241c-643a-4a97-b7f0-298170f18098', metadata={'author': 'Training example', 'subject': '(unspecified)', 'page': 0, 'total_pages': 3, 'producer': 'ReportLab PDF Library - (opensource)', 'keywords': '', 'creationdate': '2026-10-28T09:15:21+02:00', 'source': 'sample_pdfs\\remote_work_policy.pdf', 'page_label': '1', 'title': 'Remote Work Policy', 'creator': '(unspecified)', 'moddate': '2026-10-28T09:15:21+02:00', 'trapped': '/False'}, page_content='Remote Work Policy\n1. Eligibility and Weekly Schedule\nThis fictional remote work policy allows eligible employees to work remotely up to two days per week\nwith manager approval. Remote work is an arrangement for performing normal duties away from the\noffice; it does not change role expectations, working hours, or service commitments.\nEmployees and managers should agree on a regular schedule that supports customer needs, team'),
  Document(id='aec

## 8. Validate RAG with five metrics

This section evaluates the answers with an LLM judge and compares them with reference answers. Scores range from **0 (poor) to 1 (strong)**.

| Metric | What it checks |
| --- | --- |
| Context precision | Are retrieved chunks relevant to the question? |
| Context recall | Does retrieved context contain the needed information? |
| Faithfulness | Are answer claims supported by the retrieved context? |
| Answer relevance | Does the answer address the question? |
| Answer correctness | Does the answer match the reference answer? |

These are judge-based estimates, so review the examples and scores rather than treating them as ground truth.

In [9]:
import json
import re
import pandas as pd

# Ask the judge for JSON, then validate the values with regular Python.
judge = ChatOpenAI(model=CHAT_MODEL, temperature=0)

METRIC_KEYS = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_relevance",
    "answer_correctness",
]

def parse_metric_scores(raw_output):
    """Parse judge JSON and check required fields and score ranges."""
    if hasattr(raw_output, "content"):
        raw_output = raw_output.content
    if isinstance(raw_output, list):
        raw_output = "\n".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in raw_output
        )

    text = str(raw_output).strip()
    # Accept plain JSON or JSON wrapped in a Markdown code fence.
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        text = match.group(1)
    scores = json.loads(text)
    if not isinstance(scores, dict):
        raise ValueError("Judge response must be a JSON object.")

    missing = [key for key in METRIC_KEYS + ["reason"] if key not in scores]
    if missing:
        raise ValueError(f"Judge response is missing fields: {missing}")

    cleaned = {}
    for key in METRIC_KEYS:
        try:
            value = float(scores[key])
        except (TypeError, ValueError):
            raise ValueError(f"Judge field {key!r} must be numeric.")
        if not 0 <= value <= 1:
            raise ValueError(f"Judge field {key!r} must be between 0 and 1; got {value}.")
        cleaned[key] = value
    cleaned["reason"] = str(scores["reason"])
    return cleaned


### Prepare questions and reference answers

The reference answers below come from the five PDFs. The evaluation set is small so it is easy to inspect and keeps API use modest.

In [10]:
evaluation_set = [
    {
        "question": "How many paid annual leave days do full-time employees receive?",
        "reference": "They receive 18 days of paid annual leave per calendar year.",
    },
    {
        "question": "When should employees submit expense claims?",
        "reference": "Within 30 days of the purchase, with itemized receipts and the required business details.",
    },
    {
        "question": "What should an employee do after receiving a suspected phishing message?",
        "reference": "Do not click links or open attachments; report it to the IT help desk using the approved reporting method.",
    },
    {
        "question": "How many days per week may an employee work remotely under the example policy?",
        "reference": "Up to two days per week, with manager approval.",
    },
    {
        "question": "What approval is needed for a single expense above 10,000 rupees?",
        "reference": "Manager approval is required.",
    },
]

### Run the evaluation

In [11]:
def score_one(question, reference):
    result = rag_chain.invoke({"input": question})
    context = "\n\n".join(doc.page_content for doc in result["context"])

    judge_prompt = f"""You are a careful evaluator of a retrieval-augmented QA system.
Return only a valid JSON object with these exact keys: context_precision, context_recall, faithfulness, answer_relevance, answer_correctness, reason.
The five metric values must be numbers from 0 to 1. The reason must be one short explanation. Do not include Markdown or extra text.

Score each metric from 0 to 1. Use 1 for fully successful, 0 for unsuccessful, and intermediate values for partial success.

Context precision: relevance of retrieved context to the question.
Context recall: whether retrieved context contains the information needed by the reference answer.
Faithfulness: whether each factual claim in the generated answer is supported by the context.
Answer relevance: whether the answer directly addresses the question.
Answer correctness: whether the answer is substantively consistent with the reference answer.

Question: {question}
Reference answer: {reference}
Retrieved context: {context}
Generated answer: {result['answer']}
"""
    scores = parse_metric_scores(judge.invoke(judge_prompt))
    return {
        "Question": question,
        "Reference answer": reference,
        "Generated answer": result["answer"],
        "Context precision": scores["context_precision"],
        "Context recall": scores["context_recall"],
        "Faithfulness": scores["faithfulness"],
        "Answer relevance": scores["answer_relevance"],
        "Answer correctness": scores["answer_correctness"],
        "Judge note": scores["reason"],
    }

rows = [score_one(item["question"], item["reference"]) for item in evaluation_set]
results_df = pd.DataFrame(rows)
display(results_df)

,Question,Reference answer,Generated answer,Context precision,Context recall,Faithfulness,Answer relevance,Answer correctness,Judge note
0,How many paid annual leave days do full-time e...,They receive 18 days of paid annual leave per ...,Full-time employees receive 18 days of paid an...,1.0,1.0,1.0,1.0,1.0,The retrieved context fully supports the answe...
1,When should employees submit expense claims?,"Within 30 days of the purchase, with itemized ...",Employees should submit expense claims in the ...,1.0,1.0,1.0,1.0,1.0,The retrieved context fully supports the answe...
2,What should an employee do after receiving a s...,Do not click links or open attachments; report...,"After receiving a suspected phishing message, ...",1.0,1.0,1.0,1.0,1.0,The retrieved context fully supports the answe...
3,How many days per week may an employee work re...,"Up to two days per week, with manager approval.","Under the example policy, an employee may work...",1.0,1.0,1.0,1.0,1.0,The retrieved context fully supports the answe...
4,What approval is needed for a single expense a...,Manager approval is required.,Manager approval is required for any single ex...,1.0,1.0,1.0,1.0,1.0,The retrieved context fully supports the gener...


### Average metric scores

In [12]:
metric_columns = [
    "Context precision", "Context recall", "Faithfulness",
    "Answer relevance", "Answer correctness"
]
average_scores = results_df[metric_columns].mean().round(2)
display(average_scores.to_frame("Average score (0–1)"))

,Average score (0–1)
Context precision,1.0
Context recall,1.0
Faithfulness,1.0
Answer relevance,1.0
Answer correctness,1.0


### Reading the results

A low **context precision** score suggests retrieval is bringing in irrelevant chunks. Low **context recall** suggests useful evidence was missed. Low **faithfulness** suggests the answer includes unsupported claims. Low **answer relevance** or **correctness** suggests the response needs improvement. Use the judge notes and source text to investigate low scores.

## Summary

The notebook loads five PDF files, builds a retrieval-based question-answer chain, and evaluates it with five judge-based metrics: context precision, context recall, faithfulness, answer relevance, and answer correctness.
